# Dashboard do Corpus STIL 2023

Este notebook para Google Colab importa o JSON dos artigos, agrega o corpus, calcula estatísticas textuais e exibe um dashboard com tabelas, gráficos e nuvem de palavras.

Requisitos atendidos:
- importar o corpus dos artigos
- tokenizar e quantificar
- não retirar stopwords
- quantidade de tokens e types
- quantidade de sentenças
- quantidade por classe gramatical
- quantidade de lemas
- top 10 palavras do corpus
- nuvem de palavras

In [ ]:
!pip install -q pandas plotly wordcloud matplotlib

## Etapa 1: carregar o arquivo JSON

Você pode subir o arquivo `stil2023_articles.json` gerado pelo scraper. Se preferir, também pode montar o Google Drive e apontar para o caminho do arquivo.

In [ ]:
import json
from pathlib import Path
from google.colab import files

uploaded = files.upload()
json_path = next(iter(uploaded.keys()))
print(f"Arquivo carregado: {json_path}")

## Etapa 2: importar bibliotecas e definir funções auxiliares

As funções abaixo fazem a leitura do corpus, a agregação dos campos e o cálculo das estatísticas.

In [ ]:
import json
import math
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display
from wordcloud import WordCloud


WORD_RE = re.compile(r"\w", re.UNICODE)
SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def load_articles(path: str):
    """Carrega o JSON de artigos produzido pelo scraper."""
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def sentence_count(text: str) -> int:
    """Conta sentenças de forma simples usando pontuação final."""
    text = (text or "").strip()
    if not text:
        return 0
    parts = [p.strip() for p in SENTENCE_SPLIT_RE.split(text) if p.strip()]
    return len(parts) if parts else 1


def is_word_token(token: str) -> bool:
    """Considera como palavra qualquer token com pelo menos um caractere alfanumérico."""
    return bool(WORD_RE.search(token or ""))


def build_corpus_stats(articles):
    """Agrega tokens, lemas, POS e sentenças do corpus inteiro."""
    all_tokens = []
    all_word_tokens = []
    all_pos = []
    all_lemmas = []
    sentence_total = 0
    full_texts = []

    for article in articles:
        text = article.get("artigo_completo", "") or ""
        tokens = article.get("artigo_tokenizado", []) or []
        pos_tags = article.get("pos_tagger", []) or []
        lemmas = article.get("lema", []) or []

        sentence_total += sentence_count(text)
        full_texts.append(text)
        all_tokens.extend(tokens)
        all_word_tokens.extend([token for token in tokens if is_word_token(token)])
        all_pos.extend(pos_tags)
        all_lemmas.extend([lemma for lemma in lemmas if is_word_token(lemma)])

    token_counter = Counter(all_word_tokens)
    pos_counter = Counter(all_pos)
    lemma_counter = Counter(all_lemmas)

    return {
        "articles": articles,
        "texts": full_texts,
        "all_tokens": all_tokens,
        "all_word_tokens": all_word_tokens,
        "all_pos": all_pos,
        "all_lemmas": all_lemmas,
        "sentence_total": sentence_total,
        "token_counter": token_counter,
        "pos_counter": pos_counter,
        "lemma_counter": lemma_counter,
        "token_total": len(all_word_tokens),
        "type_total": len(set(token.casefold() for token in all_word_tokens)),
        "lemma_total": len(set(lemma.casefold() for lemma in all_lemmas)),
    }


def top_words_df(stats, n=10):
    """Retorna o top N de palavras do corpus sem remover stopwords."""
    rows = []
    for token, freq in stats["token_counter"].most_common(n):
        rows.append({"palavra": token, "frequencia": freq})
    return pd.DataFrame(rows)


def pos_df(stats):
    """Retorna a distribuição de classes gramaticais."""
    rows = [{"classe_gramatical": tag, "frequencia": freq} for tag, freq in stats["pos_counter"].most_common()]
    return pd.DataFrame(rows)


def article_df(articles):
    """Cria uma tabela resumida por artigo."""
    rows = []
    for article in articles:
        rows.append({
            "titulo": article.get("titulo", ""),
            "idioma": article.get("idioma", ""),
            "tokens": len([t for t in article.get("artigo_tokenizado", []) if is_word_token(t)]),
            "types": len(set((t.casefold() for t in article.get("artigo_tokenizado", []) if is_word_token(t)))),
            "sentencas": sentence_count(article.get("artigo_completo", "") or ""),
            "lemmas": len(set((l.casefold() for l in article.get("lema", []) if is_word_token(l)))),
        })
    return pd.DataFrame(rows)


def make_summary_cards(stats, articles):
    """Exibe cartões HTML com os indicadores principais do corpus."""
    cards = [
        ("Artigos", len(articles)),
        ("Tokens", f"{stats['token_total']:,}"),
        ("Types", f"{stats['type_total']:,}"),
        ("Sentenças", f"{stats['sentence_total']:,}"),
        ("Lemmas", f"{stats['lemma_total']:,}"),
    ]

    html = """
    <div style='display:flex; gap:16px; flex-wrap:wrap; margin:16px 0;'>
    """
    for label, value in cards:
        html += f"""
        <div style='min-width:170px; padding:18px; border-radius:14px; background:#f4f7fb; border:1px solid #dbe4f0;'>
          <div style='font-size:14px; color:#4c5a67; margin-bottom:6px;'>{label}</div>
          <div style='font-size:28px; font-weight:700; color:#17324d;'>{value}</div>
        </div>
        """
    html += "</div>"
    display(HTML(html))


def plot_top_words(stats, n=10):
    """Plota o top N de palavras do corpus."""
    df = top_words_df(stats, n=n)
    fig = px.bar(df, x="palavra", y="frequencia", title=f"Top {n} palavras do corpus", text_auto=True)
    fig.update_layout(xaxis_title="Palavra", yaxis_title="Frequência")
    fig.show()


def plot_pos_distribution(stats):
    """Plota a distribuição das classes gramaticais."""
    df = pos_df(stats)
    fig = px.bar(df, x="classe_gramatical", y="frequencia", title="Quantidade por classe gramatical", text_auto=True)
    fig.update_layout(xaxis_title="Classe gramatical", yaxis_title="Frequência")
    fig.show()


def plot_article_tokens(articles):
    """Plota a quantidade de tokens por artigo."""
    df = article_df(articles).sort_values("tokens", ascending=False)
    fig = px.bar(df, x="titulo", y="tokens", title="Quantidade de tokens por artigo")
    fig.update_layout(xaxis_title="Artigo", yaxis_title="Tokens")
    fig.show()


def plot_wordcloud(stats):
    """Gera a nuvem de palavras a partir das frequências do corpus sem remover stopwords."""
    wc = WordCloud(width=1600, height=800, background_color="white", colormap="viridis")
    wc.generate_from_frequencies(stats["token_counter"])
    plt.figure(figsize=(18, 9))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title("Nuvem de palavras do corpus STIL 2023", fontsize=18)
    plt.show()


def render_dashboard(stats):
    """Renderiza o dashboard completo com indicadores, tabelas, gráficos e wordcloud."""
    articles = stats["articles"]
    make_summary_cards(stats, articles)
    display(top_words_df(stats, n=10))
    display(pos_df(stats))
    display(article_df(articles))
    plot_top_words(stats, n=10)
    plot_pos_distribution(stats)
    plot_article_tokens(articles)
    plot_wordcloud(stats)

## O que cada função faz

- `load_articles(path)`: lê o arquivo JSON e retorna a lista de artigos.
- `sentence_count(text)`: estima a quantidade de sentenças do texto.
- `is_word_token(token)`: filtra tokens para considerar apenas palavras, removendo pontuação isolada.
- `build_corpus_stats(articles)`: agrega o corpus inteiro e calcula os contadores principais.
- `top_words_df(stats, n)`: monta a tabela do top N palavras do corpus, sem retirar stopwords.
- `pos_df(stats)`: monta a tabela de frequência por classe gramatical.
- `article_df(articles)`: gera uma tabela resumida por artigo.
- `make_summary_cards(stats, articles)`: cria cartões com os indicadores centrais do corpus.
- `plot_top_words(stats, n)`: desenha o gráfico do top N palavras.
- `plot_pos_distribution(stats)`: desenha o gráfico das classes gramaticais.
- `plot_article_tokens(articles)`: mostra a distribuição de tokens por artigo.
- `plot_wordcloud(stats)`: gera a nuvem de palavras do corpus.
- `render_dashboard(stats)`: junta tudo e renderiza o dashboard completo.

## Etapa 3: construir o corpus agregado

In [ ]:
articles = load_articles(json_path)
stats = build_corpus_stats(articles)

print(f"Artigos carregados: {len(articles)}")
print(f"Tokens: {stats['token_total']}")
print(f"Types: {stats['type_total']}")
print(f"Sentenças: {stats['sentence_total']}")
print(f"Lemmas: {stats['lemma_total']}")

## Etapa 4: renderizar o dashboard

In [ ]:
render_dashboard(stats)

## Etapa 5: exportar as tabelas, se quiser

Estas células salvam as estatísticas em CSV para download posterior.

In [ ]:
top_words_df(stats, n=10).to_csv("top10_palavras.csv", index=False, encoding="utf-8")
pos_df(stats).to_csv("classes_gramaticais.csv", index=False, encoding="utf-8")
article_df(articles).to_csv("estatisticas_por_artigo.csv", index=False, encoding="utf-8")

print("Arquivos CSV gerados com sucesso.")

## Etapa 6: matriz termo x termo e frequencias de n-gramas

Esta etapa cria frequencias de unigramas, bigramas e trigramas. Para a matriz termo x termo, os bigramas sao representados como `termo_atual x proximo_termo`. Para trigramas, a matriz usa `termo_1 termo_2 x termo_3`, ou seja, o contexto de duas palavras apontando para a proxima palavra.

In [ ]:
import random
from collections import defaultdict


def repair_encoding(value):
    """Corrige casos comuns de mojibake antes de montar n-gramas."""
    if not value or not any(marker in value for marker in ("Ã", "Â", "â")):
        return value
    try:
        repaired = value.encode("latin1").decode("utf-8")
        return repaired if repaired.count("Ã") < value.count("Ã") else value
    except Exception:
        return value


def corpus_word_tokens(articles, field="artigo_tokenizado", lowercase=True):
    """Retorna os tokens-palavra do corpus, preservando a ordem dos artigos."""
    tokens = []
    for article in articles:
        article_tokens = article.get(field, []) or []
        for token in article_tokens:
            token = repair_encoding(token)
            if is_word_token(token):
                tokens.append(token.casefold() if lowercase else token)
    return tokens


def ngram_counter(tokens, n):
    """Conta n-gramas de tamanho n."""
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def ngram_frequency_df(counter, n, top=None):
    """Converte um contador de n-gramas em DataFrame ordenado por frequencia."""
    rows = []
    for gram, freq in counter.most_common(top):
        rows.append({
            "ngrama": " ".join(gram),
            "frequencia": freq,
            **{f"termo_{i + 1}": term for i, term in enumerate(gram)},
        })
    return pd.DataFrame(rows)


def bigram_matrix(tokens, min_frequency=2, max_terms=60):
    """Cria matriz termo x termo com frequencias de co-ocorrencia adjacente."""
    bigrams = ngram_counter(tokens, 2)
    frequent_terms = [term for term, _ in Counter(tokens).most_common(max_terms)]
    matrix = pd.DataFrame(0, index=frequent_terms, columns=frequent_terms)
    for (left, right), freq in bigrams.items():
        if freq >= min_frequency and left in matrix.index and right in matrix.columns:
            matrix.loc[left, right] = freq
    return matrix


def trigram_context_matrix(tokens, min_frequency=2, max_contexts=60, max_next_terms=60):
    """Cria matriz contexto bigrama x proximo termo com frequencias de trigramas."""
    trigrams = ngram_counter(tokens, 3)
    context_counter = Counter((a, b) for (a, b, _), freq in trigrams.items() for _ in range(freq))
    next_counter = Counter(c for (_, _, c), freq in trigrams.items() for _ in range(freq))
    contexts = [" ".join(context) for context, _ in context_counter.most_common(max_contexts)]
    next_terms = [term for term, _ in next_counter.most_common(max_next_terms)]
    matrix = pd.DataFrame(0, index=contexts, columns=next_terms)
    for (a, b, c), freq in trigrams.items():
        context = f"{a} {b}"
        if freq >= min_frequency and context in matrix.index and c in matrix.columns:
            matrix.loc[context, c] = freq
    return matrix


tokens_lm = corpus_word_tokens(articles)
unigram_counts = ngram_counter(tokens_lm, 1)
bigram_counts = ngram_counter(tokens_lm, 2)
trigram_counts = ngram_counter(tokens_lm, 3)

unigram_df = ngram_frequency_df(unigram_counts, 1, top=30)
bigram_df = ngram_frequency_df(bigram_counts, 2, top=30)
trigram_df = ngram_frequency_df(trigram_counts, 3, top=30)
bigram_cooccurrence_matrix = bigram_matrix(tokens_lm, min_frequency=2, max_terms=60)
trigram_cooccurrence_matrix = trigram_context_matrix(tokens_lm, min_frequency=2, max_contexts=60, max_next_terms=60)

print(f"Tokens usados no modelo: {len(tokens_lm)}")
print(f"Unigramas distintos: {len(unigram_counts)}")
print(f"Bigramas distintos: {len(bigram_counts)}")
print(f"Trigramas distintos: {len(trigram_counts)}")

In [ ]:
display(unigram_df)
display(bigram_df)
display(trigram_df)

display(bigram_cooccurrence_matrix)
display(trigram_cooccurrence_matrix)

## Etapa 7: geracao de linguagem com bigramas e trigramas

As funcoes abaixo treinam modelos simples de Markov. O modelo de bigramas usa uma palavra como contexto. O modelo de trigramas usa duas palavras como contexto. A geracao para quando atinge o limite de palavras ou quando nao encontra uma continuacao no corpus.

In [ ]:
def build_markov_model(tokens, order):
    """Cria um modelo de Markov de ordem 2 ou 3 a partir dos tokens."""
    if order not in {2, 3}:
        raise ValueError("Use order=2 para bigramas ou order=3 para trigramas.")
    model = defaultdict(Counter)
    context_size = order - 1
    for i in range(len(tokens) - context_size):
        context = tuple(tokens[i:i + context_size])
        next_token = tokens[i + context_size]
        model[context][next_token] += 1
    return model


def weighted_choice(counter):
    """Sorteia uma palavra proporcionalmente a frequencia no corpus."""
    terms = list(counter.keys())
    weights = list(counter.values())
    return random.choices(terms, weights=weights, k=1)[0]


def seed_tokens_from_sentence(sentence):
    """Tokeniza uma sentenca inicial de forma simples para usar como semente."""
    sentence = repair_encoding(sentence)
    return [token.casefold() for token in re.findall(r"\w+", sentence, flags=re.UNICODE)]


def generate_text(model, seed_sentence, order, max_words=40):
    """Gera texto a partir de uma sentenca inicial usando bigramas ou trigramas."""
    generated = seed_tokens_from_sentence(seed_sentence)
    context_size = order - 1
    if len(generated) < context_size:
        generated = tokens_lm[:context_size] + generated

    for _ in range(max_words):
        context = tuple(generated[-context_size:])
        if context not in model:
            if order == 3 and len(context) == 2 and (context[-1],) in bigram_model:
                next_token = weighted_choice(bigram_model[(context[-1],)])
            elif order == 2 and unigram_counts:
                next_token = weighted_choice(unigram_counts)
            elif order == 3 and unigram_counts:
                next_token = weighted_choice(unigram_counts)
            else:
                break
        else:
            next_token = weighted_choice(model[context])
        generated.append(next_token)
    return " ".join(generated)


def random_corpus_sentence(articles):
    """Sorteia uma sentenca do corpus para usar como ponto de partida."""
    sentence_candidates = []
    for article in articles:
        text = repair_encoding(article.get("artigo_completo", "") or "")
        sentence_candidates.extend([s.strip() for s in SENTENCE_SPLIT_RE.split(text) if len(s.split()) >= 4])
    return random.choice(sentence_candidates) if sentence_candidates else "A linguagem natural e"


random.seed(42)
bigram_model = build_markov_model(tokens_lm, order=2)
trigram_model = build_markov_model(tokens_lm, order=3)

sentenca_aleatoria = random_corpus_sentence(articles)
sentenca_semantica = "A semântica é"

print("Sentenca aleatoria:")
print(sentenca_aleatoria)
print("\nGeracao com bigramas a partir da sentenca aleatoria:")
print(generate_text(bigram_model, sentenca_aleatoria, order=2, max_words=40))
print("\nGeracao com trigramas a partir da sentenca aleatoria:")
print(generate_text(trigram_model, sentenca_aleatoria, order=3, max_words=40))

print("\nGeracao com bigramas a partir de 'A semântica é...':")
print(generate_text(bigram_model, sentenca_semantica, order=2, max_words=40))
print("\nGeracao com trigramas a partir de 'A semântica é...':")
print(generate_text(trigram_model, sentenca_semantica, order=3, max_words=40))

## Etapa 8: exportar matrizes e n-gramas

Esta celula salva as tabelas geradas nesta etapa em CSV para entrega ou analise posterior.

In [ ]:
unigram_df.to_csv("unigramas_top30.csv", index=False, encoding="utf-8")
bigram_df.to_csv("bigramas_top30.csv", index=False, encoding="utf-8")
trigram_df.to_csv("trigramas_top30.csv", index=False, encoding="utf-8")
bigram_cooccurrence_matrix.to_csv("matriz_coocorrencia_bigramas.csv", encoding="utf-8")
trigram_cooccurrence_matrix.to_csv("matriz_coocorrencia_trigramas.csv", encoding="utf-8")

print("Arquivos de n-gramas e matrizes gerados com sucesso.")